# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [12]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/random_olmo_other_olmo/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [13]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/random_olmo_other_olmo/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: []
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/random_olmo_other_olmo/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: []
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/random_olmo_other_olmo/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: []


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [14]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                \
                       base             base_inv                       ft   
0           .Today (0.0239)      urrenc (0.0217)          .Today (0.0266)   
1          .Second (0.0114)       pos (5.16e-03)       .Second (8.61e-03)   
2        Buccane (8.85e-03)       act (4.85e-03)       Buccane (6.71e-03)   
3          /Area (6.07e-03)    askell (3.54e-03)          fter (5.58e-03)   
4            .au (4.88e-03)      gons (3.33e-03)     /problems (5.40e-03)   
5      /problems (4.03e-03)        �� (2.09e-03)         /Area (5.25e-03)   
6            aru (3.91e-03)      ejec (2.01e-03)      /problem (4.76e-03)   
7      /entities (2.96e-03)      azon (1.95e-03)           .au (3.83e-03)   
8       /problem (2.69e-03)        دي (1.95e-03)     /entities (3.39e-03)   
9          /bind (2.69e-03)     fácil (1.79e-03)         /Math (3.17e-03)   
10         /Math (2.61e-03)      anth (1.73e-03)           aru (2.99e-03)   
11      /respond (2.61e-03)     posix (1.73e-03)           eft (2.81e-03)   
12          fter (2.46e-03)  essional (1.63e-03)      /respond (2.81e-03)   
13    confidence (2.30e-03)  Optional (1.57e-03)        soever (2.64e-03)   
14     /operator (2.23e-03)      Vers (1.48e-03)     /activity (2.47e-03)   
15   persistence (2.23e-03)    Phones (1.43e-03)   persistence (2.40e-03)   
16     /activity (2.17e-03)        vs (1.39e-03)     /operator (2.32e-03)   
17          ilot (1.97e-03)       med (1.27e-03)         /bind (2.26e-03)   
18     belonging (1.97e-03)      orst (1.27e-03)           ERM (2.12e-03)   
19          oire (1.85e-03)  >Welcome (1.27e-03)            ют (1.98e-03)   

                                                                      \
                 ft_inv                 diff                diff_inv   
0       urrenc (0.0204)         ött (0.0262)         mens (7.69e-03)   
1        act (4.55e-03)     ursal (9.64e-03)         clip (6.99e-03)   
2        pos (4.27e-03)  populate (6.26e-03)          ans (6.77e-03)   
3       gons (3.11e-03)       Gle (6.26e-03)          ush (4.97e-03)   
4     askell (2.75e-03)        MS (5.86e-03)            � (4.12e-03)   
5       anth (2.08e-03)       iji (5.86e-03)         sequ (4.00e-03)   
6   essional (2.08e-03)    Wonder (5.86e-03)           ​​ (3.86e-03)   
7         دي (2.01e-03)     Guess (5.19e-03)          lda (3.63e-03)   
8      fácil (1.95e-03)       OND (4.30e-03)      nothing (3.42e-03)   
9         �� (1.89e-03)     ambio (3.34e-03)   comparator (3.31e-03)   
10       med (1.72e-03)     Guess (3.34e-03)         ires (3.11e-03)   
11      azon (1.62e-03)     guard (2.94e-03)         uren (3.11e-03)   
12      ejec (1.57e-03)   stories (2.94e-03)          SIM (2.91e-03)   
13      Vers (1.52e-03)    ordova (2.61e-03)         uste (2.91e-03)   
14    Phones (1.47e-03)      afia (2.61e-03)          ckt (2.82e-03)   
15        vs (1.47e-03)     ienen (2.44e-03)           àn (2.58e-03)   
16     posix (1.38e-03)   entanyl (2.44e-03)          .te (2.50e-03)   
17        bi (1.34e-03)   criptor (2.44e-03)      Passage (2.50e-03)   
18         次 (1.22e-03)       uba (2.15e-03)      private (2.43e-03)   
19       ')" (1.14e-03)     queda (2.15e-03)          ][- (2.43e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.9219)   
1           The (0.0452)      contador (0.1309)          The (0.0356)   
2           Let (0.0156)           메 (8.36e-03)           In (0.0140)   
3            In (0.0138)         иск (3.49e-03)          Let (0.0103)   
4         ### (4.49e-03)     Produto (2.12e-03)        ### (5.46e-03)   
5           A (2.88e-03)           � (1.42e-05)          A (4.00e-03)   
6         For (1.28e-03)      Resets (1.11e-05)        For (1.14e-03)   
7          As (9.99e-04)     Detalle (9.83e-06)         As (9.54e-04)   
8         

In [15]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0430)   
1        /entities (0.0265)       (us (5.07e-03)       /entities (0.0278)   
2        /problems (0.0171)      sagt (4.46e-03)       /problems (0.0192)   
3         .Today (9.16e-03)       ]]; (3.94e-03)        .Today (8.73e-03)   
4        /global (8.61e-03)        że (3.48e-03)       /global (8.24e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)       /manage (7.48e-03)   
6           /job (6.68e-03)       ($. (2.70e-03)          /job (6.81e-03)   
7   /preferences (6.10e-03)      -ves (2.70e-03)       /layout (6.41e-03)   
8        /layout (5.74e-03)       ')" (2.70e-03)  /preferences (5.83e-03)   
9      /provider (5.07e-03)     zeigt (2.55e-03)     /provider (5.46e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)   /connection (4.55e-03)   
11   /connection (4.18e-03)     feliz (2.24e-03)       /crypto (4.39e-03)   
12    WHATSOEVER (4.06e-03)     lesen (2.11e-03)    WHATSOEVER (4.27e-03)   
13  /environment (4.06e-03)       (!! (1.98e-03)          /reg (3.88e-03)   
14      /logging (3.94e-03)   kontrol (1.98e-03)  /environment (3.77e-03)   
15       /engine (3.81e-03)    spiele (1.98e-03)      /effects (3.77e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)       /engine (3.77e-03)   
17       /entity (3.48e-03)     scrut (1.54e-03)      /logging (3.77e-03)   
18       /dialog (3.37e-03)       )": (1.45e-03)       /dialog (3.77e-03)   
19      /effects (3.37e-03)       fas (1.45e-03)       /entity (3.42e-03)   

                                                                  \
                 ft_inv                  diff           diff_inv   
0        .vn (8.36e-03)     ichtig (6.81e-03)        ​​ (0.1680)   
1        (us (4.79e-03)      spiel (6.38e-03)         � (0.0292)   
2       sagt (3.97e-03)      /exec (6.38e-03)      ador (0.0133)   
3        ]]; (3.97e-03)     fenced (6.38e-03)       row (0.0107)   
4         że (3.49e-03)    ificado (6.01e-03)    undy (9.46e-03)   
5     testim (2.90e-03)      aired (5.65e-03)    stin (7.81e-03)   
6       -ves (2.90e-03)        iou (4.39e-03)   andel (7.35e-03)   
7        ')" (2.72e-03)   ollapsed (4.12e-03)   older (6.93e-03)   
8      zeigt (2.56e-03)     itives (3.88e-03)     ron (6.29e-03)   
9      lesen (2.26e-03)      imity (3.65e-03)     oca (5.22e-03)   
10       ($. (2.26e-03)       onia (3.65e-03)    utch (4.76e-03)   
11     spons (2.12e-03)       ickt (3.42e-03)  holder (4.76e-03)   
12     feliz (2.00e-03)  itionally (3.22e-03)     eng (3.81e-03)   
13    spiele (1.87e-03)        Ter (3.22e-03)   stein (3.37e-03)   
14       (!! (1.87e-03)      yecto (3.02e-03)    adel (3.37e-03)   
15   kontrol (1.75e-03)      serta (3.02e-03)     aff (3.07e-03)   
16       )": (1.65e-03)       ynom (2.84e-03)    okin (2.47e-03)   
17       fas (1.65e-03)         Zu (2.84e-03)    vest (2.40e-03)   
18     scrut (1.46e-03)         +l (2.67e-03)    stan (2.32e-03)   
19      helf (1.46e-03)        SHA (2.67e-03)    sett (2.18e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5898)     contador (0.8555)        , (0.5938)   
1       and (0.1484)    kontrol (7.39e-03)      and (0.1572)   
2       the (0.1270)         bö (5.77e-03)      the (0.1187)   
3        in (0.0552)   karakter (5.77e-03)       in (0.0564)   
4       ' ' (0.0479)         �� (5.77e-03)      ' ' (0.0430)   
5         a (0.0129)         �� (4.49e-03)        a (0.0128)   
6       ( (3.28e-03)      subur (3.49e-03)      ( (4.06e-03)   
7      to (2.99e-03)      KANJI (3.49e-03)     to (2.94e-03)   
8      of (2.79e-03)       samt (2.72e-03)     of (2.85e-03)   
9      by (2.33e-03)     testim (2.72e-03)     by (2.33e-03)   
10    for (1.68e-03)     kosten (2.12e-03)    for (1.59e-03)

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [16]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0256)         .vn (0.0196)       /entities (0.0272)   
1         /problem (0.0140)   /Register (0.0113)        /problem (0.0143)   
2      /problems (9.21e-03)    testim (6.99e-03)     /problems (9.95e-03)   
3        /global (6.70e-03)      sagt (6.58e-03)       /global (6.39e-03)   
4   /environment (6.00e-03)     asign (5.31e-03)   /connection (6.24e-03)   
5      /provider (5.89e-03)       -ie (4.90e-03)     /provider (6.10e-03)   
6         .Today (5.79e-03)     zeigt (4.03e-03)  /environment (5.71e-03)   
7    /connection (5.72e-03)        że (3.99e-03)        .Today (5.67e-03)   
8        /manage (5.60e-03)      -ves (3.29e-03)     /customer (5.31e-03)   
9      /customer (4.73e-03)         ť (2.94e-03)       /manage (5.18e-03)   
10  /preferences (4.25e-03)   personn (2.83e-03)  /preferences (3.92e-03)   
11       /dialog (3.37e-03)     probs (2.79e-03)       /shared (3.67e-03)   
12       /shared (3.35e-03)      elig (2.59e-03)       /dialog (3.48e-03)   
13      /account (3.20e-03)    ):\n\n (2.36e-03)      libertin (3.44e-03)   
14       /entity (3.17e-03)      roku (2.35e-03)      /account (3.17e-03)   
15      libertin (3.12e-03)     lesen (2.31e-03)       /entity (3.17e-03)   
16       /layout (2.92e-03)  ,,,,,,,, (2.23e-03)       /layout (3.13e-03)   
17          .Try (2.83e-03)       )": (2.21e-03)      /effects (2.91e-03)   
18      /effects (2.73e-03)    spiele (2.12e-03)          .Try (2.82e-03)   
19        /legal (2.64e-03)       (us (2.09e-03)         .Take (2.75e-03)   

                                                                     \
                 ft_inv                  diff              diff_inv   
0          .vn (0.0202)        spiel (0.0242)           ​​ (0.1084)   
1    /Register (0.0109)         ickt (0.0121)            � (0.0135)   
2     testim (6.92e-03)     ighted (7.41e-03)        row (7.06e-03)   
3       sagt (6.03e-03)        éis (7.32e-03)        ron (6.52e-03)   
4      asign (5.17e-03)     ichtig (7.30e-03)      andle (5.20e-03)   
5        -ie (4.82e-03)      ibili (7.18e-03)       ador (5.07e-03)   
6      zeigt (4.04e-03)      Regel (5.70e-03)        ens (3.64e-03)   
7         że (3.83e-03)      aired (5.67e-03)       stin (3.22e-03)   
8       -ves (3.24e-03)  reachable (5.56e-03)      older (3.19e-03)   
9    personn (2.98e-03)       anus (5.36e-03)       itch (2.94e-03)   
10         ť (2.97e-03)         äß (5.34e-03)      enser (2.51e-03)   
11     probs (2.90e-03)      imity (5.19e-03)      andel (2.49e-03)   
12      elig (2.50e-03)       cine (3.67e-03)      (...) (2.39e-03)   
13       )": (2.34e-03)         ền (3.65e-03)       usan (2.36e-03)   
14     lesen (2.30e-03)     itives (3.65e-03)   powerful (2.34e-03)   
15    ):\n\n (2.27e-03)     ibilit (3.38e-03)        red (2.31e-03)   
16      roku (2.22e-03)       iado (3.36e-03)      chten (2.19e-03)   
17    spiele (2.18e-03)         ôm (3.35e-03)       vest (2.16e-03)   
18  ,,,,,,,, (2.10e-03)   ollapsed (3.16e-03)       wich (1.91e-03)   
19       esl (2.01e-03)     aturas (2.85e-03)         ージ (1.86e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8030)     contador (0.9620)         , (0.8253)   
1        ' ' (0.1082)    kontrol (3.15e-03)       ' ' (0.0882)   
2        the (0.0407)   karakter (2.50e-03)       the (0.0386)   
3        and (0.0302)       rekl (1.60e-03)       and (0.0307)   
4       in (5.89e-03)         bö (1.38e-03)      in (5.81e-03)   
5        ( (4.36e-03)       zoek (1.15e-03)       ( (5.53e-03)   
6       's (2.98e-03)    Produto (9.77e-04)      's (1.69e-03)   
7        a (1.66e-03)     testim (9.74e-04)       a (1.54e-03)   
8       to (9.75e-04)     bilder (8.70e-04)      to (8.99e-04)   
9        . (6.60e-04)         �� (7.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [17]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/model-organisms/diffing_results/olmo2_1B/random_olmo_other_olmo/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture/base_auto_patch_scope_pos_-1_openai_gpt-5-mini.pt'

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [ ]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [18]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                             \
                    pos_-3               pos_-1                 pos_0   
0            ícia (0.0208)         ött (0.0262)        ursal (0.0273)   
1         Masters (0.0111)     ursal (9.64e-03)         xima (0.0256)   
2           ite (9.77e-03)  populate (6.26e-03)         cita (0.0212)   
3          ceso (9.22e-03)       Gle (6.26e-03)        yecto (0.0212)   
4          utan (6.71e-03)        MS (5.86e-03)         haps (0.0137)   
5        ención (6.32e-03)       iji (5.86e-03)         pite (0.0107)   
6          Lies (5.92e-03)    Wonder (5.86e-03)      /exec (6.10e-03)   
7           sel (4.91e-03)     Guess (5.19e-03)     itives (5.07e-03)   
8        anking (4.61e-03)       OND (4.30e-03)      antha (4.76e-03)   
9       achuset (4.61e-03)     ambio (3.34e-03)       iltr (4.76e-03)   
10         ment (4.61e-03)     Guess (3.34e-03)       ynom (3.48e-03)   
11     Insights (4.33e-03)     guard (2.94e-03)      spiel (3.48e-03)   
12          ERY (4.09e-03)   stories (2.94e-03)       Shar (3.27e-03)   
13         Seks (3.83e-03)    ordova (2.61e-03)       ickt (3.27e-03)   
14  ryptography (3.17e-03)      afia (2.61e-03)    .rstrip (3.07e-03)   
15         iltr (2.99e-03)     ienen (2.44e-03)      endum (2.88e-03)   
16         fila (2.99e-03)   entanyl (2.44e-03)  itionally (2.70e-03)   
17         tant (2.81e-03)   criptor (2.44e-03)      mente (2.70e-03)   
18        antha (2.81e-03)       uba (2.15e-03)  vironment (2.38e-03)   
19          xec (2.64e-03)     queda (2.15e-03)      Gover (2.38e-03)   

                                                                      \
                   pos_1                 pos_2                 pos_3   
0          xima (0.0234)       ichtig (0.0125)       ichtig (0.0164)   
1        ichtig (0.0161)      imity (8.12e-03)        spiel (0.0120)   
2         ursal (0.0134)       ynom (6.32e-03)          iou (0.0106)   
3         yecto (0.0134)     fenced (6.32e-03)      aired (9.89e-03)   
4        ynom (6.32e-03)       iltr (5.55e-03)     fenced (5.65e-03)   
5       /exec (4.61e-03)       xima (5.55e-03)       ynom (5.65e-03)   
6        rega (4.61e-03)    aussian (5.22e-03)     itives (3.88e-03)   
7        pite (4.61e-03)      /exec (4.61e-03)      /exec (3.65e-03)   
8        iltr (4.09e-03)      ursal (4.06e-03)      imity (3.42e-03)   
9   itionally (4.09e-03)        iou (3.83e-03)       irse (3.02e-03)   
10    aussian (2.98e-03)      aired (3.60e-03)       ickt (2.84e-03)   
11        -li (2.62e-03)      spiel (3.17e-03)       iltr (2.67e-03)   
12     ighted (2.47e-03)       fakt (2.98e-03)      ından (2.67e-03)   
13   /student (2.47e-03)  itionally (2.98e-03)    ificado (2.67e-03)   
14     urette (2.32e-03)       utos (2.98e-03)      yecto (2.67e-03)   
15         �� (2.32e-03)    INVALID (2.79e-03)  itionally (2.50e-03)   
16        iou (2.18e-03)     agnost (2.18e-03)      serta (2.50e-03)   
17        hoe (1.92e-03)      abbix (2.18e-03)     inally (2.35e-03)   
18    recurse (1.92e-03)    initely (2.18e-03)    initely (2.21e-03)   
19    [];\n\n (1.92e-03)         +l (2.04e-03)       iniz (2.21e-03)   

                                                                      \
                  pos_10                pos_50               pos_100   
0         imity (0.0123)        spiel (0.0325)        spiel (0.0288)   
1         spiel (0.0109)         ickt (0.0112)         ickt (0.0186)   
2        ichtig (0.0109)        éis (8.24e-03)       ighted (0.0113)   
3       aired (9.58e-03)  reachable (7.72e-03)      ibili (9.95e-03)   
4      itives (7.45e-03)      Regel (7.72e-03)        éis (8.79e-03)   
5        ickt (5.13e-03)      ibili (7.23e-03)       anus (8.24e-03)   
6     ificado (5.13e-03)     ighted (7.23e-03)         äß (7.75e-03)   
7       /exec (4.52e-03)     ichtig (6.81e-03)      Regel (5.68e-03)   
8      ibilit (4.24e-03)         äß (6.41e-03)      aired (5.34e-03)   
9   reachable (4.24e-03)      aired

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [ ]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()